In [0]:
# ==============================================================================
# RAGシステム メインノートブック
# Databricks 16.4 LTS対応版
# ==============================================================================

# セル1: 環境設定
# ------------------------------------------------------------------------------
# オートリロード設定（pyファイル更新時の自動リロード）
%load_ext autoreload
%autoreload 2

# 警告抑制
import warnings
warnings.filterwarnings('ignore')

print("✅ 環境設定完了")

In [0]:
# セル2: 新規ライブラリインストール
# ------------------------------------------------------------------------------
# 既存ライブラリ保護のため、新規ライブラリのみインストール
# 注意: 既存のlangchain 0.3.21、pydantic 2.8.2等は絶対に再インストールしない

print("🔄 新規ライブラリのインストールを開始します...")
print("⚠️ 既存ライブラリ（langchain 0.3.21、pydantic 2.8.2等）は保護されます")

# LangChain拡張ライブラリ
%pip install langchain-openai==0.2.9
%pip install langchain-community==0.3.21

# ドキュメント処理ライブラリ
%pip install python-docx==1.1.2
%pip install openpyxl==3.1.5
%pip install python-pptx==1.0.2

# 検索・ベクトル処理ライブラリ
%pip install faiss-cpu==1.8.0
%pip install rank-bm25==0.2.2

print("✅ 新規ライブラリインストール完了")
print("🔄 Pythonランタイムを再起動します...")

# ライブラリインストール完了後の再起動
dbutils.library.restartPython()


In [0]:
# セル3: ライブラリインポート
# ------------------------------------------------------------------------------
print("📚 ライブラリインポートを開始します...")

# 既存ライブラリ（絶対保護対象）
try:
    import langchain
    import langchain_core
    import langchain_text_splitters
    import pydantic
    print(f"✅ 既存ライブラリ確認 - LangChain: {langchain.__version__}")
    print(f"✅ 既存ライブラリ確認 - Pydantic: {pydantic.__version__}")
except ImportError as e:
    print(f"❌ 既存ライブラリインポートエラー: {e}")

# 新規インストールライブラリ
try:
    import langchain_openai
    import langchain_community
    print("✅ 新規ライブラリインポート完了")
except ImportError as e:
    print(f"❌ 新規ライブラリインポートエラー: {e}")

# Databricks標準ライブラリ
import mlflow
import pandas as pd
import numpy as np
print("✅ Databricks標準ライブラリインポート完了")

# 標準ライブラリ
import os
import json
import pickle
from datetime import datetime
print("✅ 標準ライブラリインポート完了")

# カスタムクラス
try:
    from llm_base import LLMBase
    from rag_builder import RAGBuilder
    from rag_retriever import RAGRetriever
    print("✅ カスタムクラスインポート完了")
except ImportError as e:
    print(f"❌ カスタムクラスインポートエラー: {e}")
    print("⚠️ 先にllm_base.py、rag_builder.py、rag_retriever.pyファイルを作成してください")

print("\n" + "="*60)
print("🎯 RAGシステム準備完了")
print("="*60)



In [0]:
# セル4: RAGシステム構築実行
# ------------------------------------------------------------------------------
print("🚀 RAGシステム構築を開始します...")
print("📁 処理対象: /Volumes/koiso_databircks_16/b_0048/vol/input_file")
print("-" * 60)

# RAG構築インスタンス作成
try:
    builder = RAGBuilder(dbutils)
    print("✅ RAGBuilderインスタンス作成完了")
except Exception as e:
    print(f"❌ RAGBuilderインスタンス作成エラー: {e}")
    raise

# RAGシステム構築実行
construction_start_time = datetime.now()

try:
    print("\n🔄 RAGシステム構築実行中...")
    build_success = builder.build_rag_system()
    
    construction_time = (datetime.now() - construction_start_time).total_seconds()
    
    if build_success:
        print(f"\n✅ RAGシステム構築が完了しました")
        print(f"⏰ 構築時間: {construction_time:.2f}秒")
        
        # 成果物確認機能の自動実行
        print("\n" + "="*60)
        print("🔍 成果物確認を実行します...")
        print("="*60)
        
        confirmation_results = builder.confirm_build_results()
        
        if confirmation_results:
            print("\n✅ 成果物確認が完了しました")
            print("📊 構築結果の詳細は上記の確認結果を参照してください")
        else:
            print("\n⚠️ 成果物確認でエラーが発生しました")
            
    else:
        print(f"\n❌ RAGシステム構築に失敗しました")
        print(f"⏰ 経過時間: {construction_time:.2f}秒")
        
except Exception as e:
    construction_time = (datetime.now() - construction_start_time).total_seconds()
    print(f"\n❌ RAGシステム構築エラー: {e}")
    print(f"⏰ 経過時間: {construction_time:.2f}秒")
    raise

print("\n" + "="*60)
print("🏗️ RAGシステム構築フェーズ完了")
print("="*60)



In [0]:
# セル5: 検索・回答生成システム初期化
# ------------------------------------------------------------------------------
print("🔄 検索・回答生成システムを初期化します...")

# 回答生成インスタンス作成
try:
    retriever = RAGRetriever(dbutils)
    print("✅ RAGRetrieverインスタンス作成完了")
except Exception as e:
    print(f"❌ RAGRetrieverインスタンス作成エラー: {e}")
    raise

# 検索システム初期化
try:
    initialization_success = retriever.initialize_retrieval_system()
    
    if initialization_success:
        print("✅ 検索・回答生成システムの初期化が完了しました")
        print("\n📋 システム状態:")
        print(f"  - Faissインデックス: {'✅ 読み込み済み' if retriever.faiss_index else '❌ 未読み込み'}")
        print(f"  - BM25インデックス: {'✅ 読み込み済み' if retriever.bm25_index else '❌ 未読み込み'}")
        print(f"  - ドキュメント数: {len(retriever.documents) if retriever.documents else 0}")
        print(f"  - 会話メモリ: {'✅ 初期化済み' if retriever.conversation_memory else '❌ 未初期化'}")
    else:
        print("❌ 検索・回答生成システムの初期化に失敗しました")
        print("⚠️ 先にRAGシステム構築が正常に完了していることを確認してください")
        
except Exception as e:
    print(f"❌ 初期化エラー: {e}")
    raise

print("\n" + "="*60)
print("🎯 検索・回答生成システム準備完了")
print("="*60)



In [0]:
# セル6: 対話実行（メインループ）
# ------------------------------------------------------------------------------
print("🤖 RAGシステムが準備完了しました。質問をどうぞ！")
print("💡 ヒント:")
print("  - 各質問で検索スコアが自動表示されます")
print("  - 会話履歴が自動的に保持されます")
print("  - 終了する場合は 'quit' または 'exit' と入力してください")
print("\n" + "="*60)

# 対話統計
conversation_count = 0
session_start_time = datetime.now()

while True:
    try:
        # ユーザー入力
        print(f"\n💭 質問 #{conversation_count + 1}:")
        question = input(">>> ")
        
        # 終了チェック
        if question.lower().strip() in ['quit', 'exit', '終了', 'q']:
            session_duration = (datetime.now() - session_start_time).total_seconds()
            print(f"\n👋 お疲れ様でした！")
            print(f"📊 セッション統計:")
            print(f"  - 質問数: {conversation_count}件")
            print(f"  - セッション時間: {session_duration:.1f}秒")
            break
            
        # 空の質問チェック
        if not question.strip():
            print("⚠️ 質問を入力してください")
            continue
        
        # 回答生成（スコア表示を含む）
        print(f"\n🔍 検索中...")
        response_start_time = datetime.now()
        
        response = retriever.generate_response(question)
        
        response_time = (datetime.now() - response_start_time).total_seconds()
        conversation_count += 1
        
        # 回答表示
        print(f"\n🤖 回答:")
        print("-" * 60)
        print(response)
        print("-" * 60)
        print(f"⏰ 回答生成時間: {response_time:.2f}秒")
        
    except KeyboardInterrupt:
        print(f"\n\n👋 キーボード割り込みで終了します")
        session_duration = (datetime.now() - session_start_time).total_seconds()
        print(f"📊 セッション統計: {conversation_count}件の質問, {session_duration:.1f}秒")
        break
        
    except Exception as e:
        print(f"\n❌ エラーが発生しました: {e}")
        print("🔄 もう一度お試しください")

print("\n" + "="*60)
print("🎯 対話セッション終了")
print("="*60)



In [0]:
# セル7: システム状態確認（オプション）
# ------------------------------------------------------------------------------
print("📊 RAGシステム状態確認")
print("="*60)

# ディレクトリ情報
try:
    directories_info = [
        ("入力ファイル", retriever.INPUT_DIRECTORY),
        ("抽出テキスト", retriever.EXTRACTED_TEXT_DIR),
        ("チャンク", retriever.CHUNKED_TEXT_DIR),
        ("インデックス", retriever.FAISS_INDEX_DIR),
        ("会話履歴", retriever.CONVERSATION_HISTORY_DIR),
        ("構築ログ", retriever.BUILD_LOGS_DIR)
    ]
    
    print("📁 ディレクトリ状態:")
    for dir_name, dir_path in directories_info:
        if os.path.exists(dir_path):
            file_count = len(os.listdir(dir_path))
            total_size_mb = sum(
                os.path.getsize(os.path.join(dir_path, f)) 
                for f in os.listdir(dir_path)
                if os.path.isfile(os.path.join(dir_path, f))
            ) / (1024 * 1024)
            print(f"  ✅ {dir_name}: {file_count}ファイル ({total_size_mb:.2f}MB)")
        else:
            print(f"  ❌ {dir_name}: ディレクトリが存在しません")
    
    # インデックス詳細情報
    print(f"\n🔢 インデックス詳細:")
    if retriever.faiss_index:
        print(f"  ✅ Faissベクトル数: {retriever.faiss_index.ntotal:,}個")
        print(f"  ✅ ベクトル次元数: {retriever.faiss_index.d}次元")
    else:
        print(f"  ❌ Faissインデックス: 未読み込み")
    
    if retriever.bm25_index:
        print(f"  ✅ BM25インデックス: 読み込み済み")
    else:
        print(f"  ❌ BM25インデックス: 未読み込み")
    
    print(f"  📄 ドキュメント数: {len(retriever.documents):,}件")
    
    # 会話状態
    conversation_summary = retriever.get_conversation_summary()
    print(f"\n💭 会話状態:")
    if 'error' not in conversation_summary:
        print(f"  📝 質問・回答数: {conversation_summary.get('total_exchanges', 0)}回")
        print(f"  📨 総メッセージ数: {conversation_summary.get('total_messages', 0)}件")
        print(f"  💾 メモリ使用量: {conversation_summary.get('memory_length', 0):,}文字")
    else:
        print(f"  ❌ 会話状態取得エラー: {conversation_summary['error']}")
    
    # 最新の構築ログ
    if os.path.exists(retriever.BUILD_LOGS_DIR):
        log_files = [f for f in os.listdir(retriever.BUILD_LOGS_DIR) if f.startswith('build_summary_')]
        if log_files:
            latest_log = sorted(log_files)[-1]
            log_path = os.path.join(retriever.BUILD_LOGS_DIR, latest_log)
            log_data = retriever.load_json(log_path)
            
            print(f"\n📋 最新構築ログ ({latest_log}):")
            if log_data:
                print(f"  🕒 構築日時: {log_data.get('build_timestamp', 'N/A')}")
                print(f"  📄 処理ファイル数: {log_data.get('total_processed_files', 0)}件")
                print(f"  🔢 総チャンク数: {log_data.get('total_chunks', 0)}個")
                print(f"  🎯 検索準備: {'✅ 完了' if log_data.get('ready_for_retrieval', False) else '❌ 未完了'}")

except Exception as e:
    print(f"❌ システム状態確認エラー: {e}")

print("="*60)
print("🎯 システム状態確認完了")


In [0]:
# セル8: ユーティリティ機能（オプション）
# ------------------------------------------------------------------------------
print("🛠️ ユーティリティ機能")
print("="*40)

# 会話履歴リセット機能
def reset_conversation():
    """会話履歴をリセット"""
    try:
        retriever.reset_conversation()
        print("✅ 会話履歴がリセットされました")
    except Exception as e:
        print(f"❌ 会話履歴リセットエラー: {e}")

# テスト検索機能
def test_search(query="テスト"):
    """検索機能のテスト"""
    try:
        print(f"🔍 テスト検索実行: '{query}'")
        results = retriever.hybrid_search(query, top_k=3)
        retriever.display_search_scores(results)
        print("✅ テスト検索完了")
    except Exception as e:
        print(f"❌ テスト検索エラー: {e}")

# システム情報表示
def show_system_info():
    """システム情報の表示"""
    print("🖥️ システム情報:")
    print(f"  Python: {os.sys.version}")
    print(f"  LangChain: {langchain.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  NumPy: {np.__version__}")
    print(f"  MLflow: {mlflow.__version__}")

# 使用例
print("💡 利用可能なユーティリティ:")
print("  - reset_conversation(): 会話履歴リセット")
print("  - test_search('検索語'): テスト検索実行")
print("  - show_system_info(): システム情報表示")

print("\n" + "="*60)
print("🎉 RAGシステム完全準備完了")
print("="*60)